# Hansa Bread & Snacks — Returns and Sales Report Generator

Builds exactly **two workbooks** — `Bread_Report.xlsx` and `Snacks_Report.xlsx` — bread and snack data are never mixed into the same file.

Every sheet in both workbooks keeps **A/C** and **Name** so every row can be traced back to a customer. Each workbook has **one sheet per calendar month found in the data** (e.g. `July 2026`, `August 2026`, ...), plus two more sheets:
- Every month sheet — **one row per customer**, totalled for that whole month (not datewise), with TOTAL_RETURNS, TOTAL_SALES, TOTAL_400G_RETURNS, TOTAL_400G_SALES, PCT_RETURN_400G
- `All Months Total` — one row per customer with that customer's totals added up across **every month present in the data**, plus a final grand-total row across all customers
- `Datewise (All Months)` — the one sheet that keeps day-level detail: every customer/date combination in the dataset, from day 1 of the first month through the last day of data

So with the current 2-month dataset (July + August) that's **4 sheets per workbook**: `July 2026`, `August 2026`, `All Months Total`, `Datewise (All Months)`. A new month in the data just adds a new month-sheet — the "data may be huge" problem is solved by splitting **by month**, not by dumping everything into one giant sheet.

Bread quantities are converted to a common **400g-equivalent** basis before % return is calculated: `200g × 0.5`, `800g × 2`, `600g × 1.5` — every other bread column is already 400g. Snacks have no weight conversion, so their "400g" columns are just the raw totals, kept only so both workbooks share the same layout.

In [1]:
import numpy as np
import pandas as pd
import os
import openpyxl

## Config — edit paths here if needed

In [2]:
RETURNS_FILE = "data\\hansa-returns\\oct-wk3-returns.xlsx"
SALES_FILE = "sales\\October\\wk2\\OCT-INV-WK3-DATEWISE.xlsx"
OUTPUT_DIR = "reports\\october\\wk3\\"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Bread-only conversion to a common 400g-equivalent basis.
# Any bread column NOT listed here is assumed to already be 400g (factor 1.0).
BREAD_FACTORS = {
    "200G": 0.5,
    "800G": 2.0,
    "600G": 1.5,
}

# Snacks have no weight conversion -> every column stays at factor 1.0
SNACK_FACTORS = {}


## Load source files
`hansa-returns.xlsx` already separates returns into a `Breads` sheet and a `Snacks` sheet — we use exactly those two sheets to figure out which product columns belong to bread vs. snacks, so the split stays correct automatically even if products are added or removed later.

In [3]:
returns_xl = pd.ExcelFile(RETURNS_FILE)
df_bread_ret_raw = pd.read_excel(returns_xl, sheet_name="Breads")
df_snack_ret_raw = pd.read_excel(returns_xl, sheet_name="Snacks")
df_sales_raw = pd.read_excel(SALES_FILE)

print("Bread returns:", df_bread_ret_raw.shape)
print("Snack returns:", df_snack_ret_raw.shape)
print("Sales:", df_sales_raw.shape)


Bread returns: (2876, 14)
Snack returns: (2876, 9)
Sales: (6828, 23)


## Work out which columns are products (vs. meta columns like Date/Invoice/A-C)
and match each product column to its counterpart in the sales file.

Column names aren't spelled identically everywhere (`200g` vs `200G`, `O.P.` vs `O.P`), so matching is done on a normalised (uppercased, stripped) version of the name.

In [4]:
def norm(col):
    """Normalise a column name so it matches across files regardless of case/spacing."""
    return " ".join(str(col).strip().upper().split())

def norm_ac(series):
    """Normalise a customer code so 55, 55.0 and '55' all match the same customer."""
    return pd.to_numeric(series, errors="coerce").astype("Int64").astype(str)

META_COLS_RETURNS = {"A/C", "NAME", "TYPE", "INVOICE", "DATE", "QTY"}

def product_cols(df, meta_set):
    return [c for c in df.columns if norm(c) not in meta_set]

bread_products = product_cols(df_bread_ret_raw, META_COLS_RETURNS)
snack_products = product_cols(df_snack_ret_raw, META_COLS_RETURNS)

print("Bread product columns (from returns):", bread_products)
print("Snack product columns (from returns):", snack_products)


Bread product columns (from returns): ['P', 'S', 'B', '200G', 'O.P.', '800G', '600g', '4 Square', 'XTRA LOONG']
Snack product columns (from returns): ['MANDAZI', 'DOUGHNUTS', 'MANDO FAMILY PACK', 'B-BITEZ']


In [5]:
sales_col_lookup = {norm(c): c for c in df_sales_raw.columns}

def map_to_sales(product_cols_list):
    mapping, missing = {}, []
    for pc in product_cols_list:
        key = norm(pc)
        if key in sales_col_lookup:
            mapping[pc] = sales_col_lookup[key]
        else:
            missing.append(pc)
    return mapping, missing

bread_sales_map, bread_missing = map_to_sales(bread_products)
snack_sales_map, snack_missing = map_to_sales(snack_products)

if bread_missing:
    print("WARNING - bread columns with no match in the sales file (excluded from sales totals):", bread_missing)
if snack_missing:
    print("WARNING - snack columns with no match in the sales file (excluded from sales totals):", snack_missing)


## Core pipeline — builds one workbook (bread OR snacks)

In [6]:
def to_numeric_df(df, cols):
    df = df.copy()
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    return df


SUMMARY_COLS = ["TOTAL_RETURNS", "TOTAL_SALES", "TOTAL_400G_RETURNS", "TOTAL_400G_SALES", "PCT_RETURN_400G"]


def build_report(label, returns_raw, product_cols, sales_raw, sales_col_map,
                  conversion_factors, output_dir=OUTPUT_DIR):
    """
    Builds ONE Excel workbook (label = "Bread" or "Snacks") with:
      - one sheet per calendar month present in the data
        (one row PER CUSTOMER, totalled for that whole month)
      - "All Months Total" (one row per customer, totalled across every month, + one grand-total row)
      - "Datewise (All Months)" (every customer/date combo in the dataset, one continuous table)
    Every sheet keeps A/C and Name so rows can be traced back to a customer.
    Returns/sales are converted to a 400g-equivalent basis using `conversion_factors`
    (columns not listed there default to factor 1.0, i.e. already 400g).
    """
    returns_raw = returns_raw.copy()
    returns_raw["Date"] = pd.to_datetime(returns_raw["Date"]).dt.date
    returns_raw["A/C"] = norm_ac(returns_raw["A/C"])
    returns_raw = to_numeric_df(returns_raw, product_cols)

    factors = {norm(c): conversion_factors.get(norm(c), 1.0) for c in product_cols}

    # ---------- RETURNS: per customer, per date ----------
    returns_grouped = returns_raw.groupby(["Date", "A/C", "Name"], as_index=False)[product_cols].sum()
    returns_grouped["TOTAL_RETURNS"] = returns_grouped[product_cols].sum(axis=1)
    returns_grouped["TOTAL_400G_RETURNS"] = sum(
        returns_grouped[c] * factors[norm(c)] for c in product_cols
    )

    # ---------- SALES: matching columns, per customer, per date ----------
    matched_products = [c for c in product_cols if c in sales_col_map]
    sales_cols_present = [sales_col_map[c] for c in matched_products]
    rename_back = dict(zip(sales_cols_present, matched_products))

    sales_raw = sales_raw.copy()
    sales_raw["Date"] = pd.to_datetime(sales_raw["Date"]).dt.date
    sales_raw["A/C"] = norm_ac(sales_raw["A/C"])
    sales_subset = sales_raw[["Date", "A/C", "Customer"] + sales_cols_present].rename(columns=rename_back)
    sales_subset = to_numeric_df(sales_subset, matched_products)

    sales_grouped = sales_subset.groupby(["Date", "A/C", "Customer"], as_index=False)[matched_products].sum()
    sales_grouped["TOTAL_SALES"] = sales_grouped[matched_products].sum(axis=1)
    sales_grouped["TOTAL_400G_SALES"] = sum(
        sales_grouped[c] * factors[norm(c)] for c in matched_products
    )
    sales_grouped = sales_grouped.rename(columns={"Customer": "Name"})

    # ---------- combine into one master table: one row per customer per date
    # (outer join keeps a customer/date that only has a return OR only has a sale) ----------
    master = returns_grouped.merge(
        sales_grouped[["Date", "A/C", "Name", "TOTAL_SALES", "TOTAL_400G_SALES"]],
        on=["Date", "A/C"], how="outer", suffixes=("", "_sales"),
    )
    master["Name"] = master["Name"].fillna(master["Name_sales"])
    master = master.drop(columns=["Name_sales"])

    for c in product_cols + ["TOTAL_RETURNS", "TOTAL_400G_RETURNS", "TOTAL_SALES", "TOTAL_400G_SALES"]:
        master[c] = master[c].fillna(0)

    master = master.sort_values(["Date", "A/C"]).reset_index(drop=True)
    master["PCT_RETURN_400G"] = np.where(
        master["TOTAL_400G_SALES"] > 0,
        master["TOTAL_400G_RETURNS"] / master["TOTAL_400G_SALES"],
        0,
    ).round(2)

    datewise_col_order = ["Date", "A/C", "Name"] + product_cols + SUMMARY_COLS
    master = master[datewise_col_order]
    master["Month"] = pd.to_datetime(master["Date"]).dt.to_period("M")

    sum_cols = product_cols + ["TOTAL_RETURNS", "TOTAL_SALES", "TOTAL_400G_RETURNS", "TOTAL_400G_SALES"]
    monthly_col_order = ["A/C", "Name"] + sum_cols + ["PCT_RETURN_400G"]

    def add_pct(df):
        df = df.copy()
        df["PCT_RETURN_400G"] = np.where(
            df["TOTAL_400G_SALES"] > 0,
            df["TOTAL_400G_RETURNS"] / df["TOTAL_400G_SALES"],
            0,
        ).round(2)
        return df

    # ---------- one row PER CUSTOMER per MONTH (whole-month totals, not datewise) ----------
    month_customer_totals = master.groupby(["Month", "A/C", "Name"], as_index=False)[sum_cols].sum()
    month_customer_totals = add_pct(month_customer_totals)

    # ---------- All Months Total: one row per customer, summed across every month ----------
    customer_totals = master.groupby(["A/C", "Name"], as_index=False)[sum_cols].sum()
    customer_totals = add_pct(customer_totals)
    customer_totals = customer_totals.sort_values("TOTAL_RETURNS", ascending=False).reset_index(drop=True)

    grand_total = customer_totals[sum_cols].sum()
    grand_total["A/C"] = ""
    grand_total["Name"] = "TOTAL (All Customers, All Months)"
    grand_total["PCT_RETURN_400G"] = round(
        (grand_total["TOTAL_400G_RETURNS"] / grand_total["TOTAL_400G_SALES"])
        if grand_total["TOTAL_400G_SALES"] else 0, 2
    )
    all_months_df = pd.concat([customer_totals, pd.DataFrame([grand_total])], ignore_index=True)
    all_months_df = all_months_df[monthly_col_order]

    # ---------- write workbook: one sheet per month + All Months Total + Datewise (All Months) ----------
    workbook_path = os.path.join(output_dir, f"{label}_Report.xlsx")
    month_sheet_names = []
    with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
        for period, group in month_customer_totals.groupby("Month"):
            sheet_name = period.strftime("%B %Y")[:31]
            group = group.sort_values("TOTAL_RETURNS", ascending=False)
            group[monthly_col_order].to_excel(writer, sheet_name=sheet_name, index=False)
            month_sheet_names.append(sheet_name)
        all_months_df.to_excel(writer, sheet_name="All Months Total", index=False)
        master[datewise_col_order].to_excel(writer, sheet_name="Datewise (All Months)", index=False)

    all_sheets = month_sheet_names + ["All Months Total", "Datewise (All Months)"]
    print(f"[{label}] workbook written to {workbook_path} — {len(all_sheets)} sheet(s): {all_sheets}")
    return workbook_path


## Run it for Bread and for Snacks

In [7]:
bread_path = build_report(
    "Bread", df_bread_ret_raw, bread_products, df_sales_raw, bread_sales_map, BREAD_FACTORS
)

snack_path = build_report(
    "Snacks", df_snack_ret_raw, snack_products, df_sales_raw, snack_sales_map, SNACK_FACTORS
)


[Bread] workbook written to reports\october\wk3\Bread_Report.xlsx — 3 sheet(s): ['October 2026', 'All Months Total', 'Datewise (All Months)']
[Snacks] workbook written to reports\october\wk3\Snacks_Report.xlsx — 3 sheet(s): ['October 2026', 'All Months Total', 'Datewise (All Months)']
